<a href="https://colab.research.google.com/github/hUSsAin976-tech/ML-internship-at-FlyRank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Answer 1 — what one row means for this lane.**
I read `fact_content_daily_performance` at its native grain: one row = one content item, on one calendar day, for one client (`report_date + client_hash_id + content_hash_id`). I never flatten this straight into "one row = one page" — that is a choice I make myself, on purpose, when I build features in Section 3: I aggregate the daily grain up to **one row per content item for the month**, because the AI Referral lane ranks *pages*, not page-days.

**Answer 2 — which table(s).**
- `fact_content_daily_performance` (the `month=2026-03` partition only, per the iteration rule — never the `_sample` table, which is the sealed final month) for the daily GSC/GA4 signals, including `sessions_ai`.
- `dim_content`, joined on `content_hash_id`, for static content metadata (`content_type`, `word_count`) that does not change day to day.
- `dim_clients` is *not* joined into the feature frame itself, but I use it below to sanity-check availability.

**Answer 3 — time window.**
Development month: **`month=2026-03`**, a mid-panel month, per the assignment's instruction to never develop label logic on the `_sample` table (final month, June 2026) since that is the natural outcome window of any past→future label and would mean testing on my own training window. The final month stays a sealed test month I do not touch in this notebook.


In [2]:
import os, getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":      f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":      f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily_month": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:16} {n:>12,} rows")

print("\nfact_daily_month columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily_month']} LIMIT 1").df()["column_name"].tolist())
print("\ndim_content columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']} LIMIT 1").df()["column_name"].tolist())


Paste your Hugging Face READ token (hf_...): ··········
dim_clients               104 rows
dim_content           519,606 rows
fact_daily_month    9,841,378 rows

fact_daily_month columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

dim_content columns:
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cp

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Answer 4 — what I predict or rank (label / proxy).**
Same proxy as ML-03: `has_ai_sessions_month` (`SUM(sessions_ai) > 0` inside the month) is an **evidence variable**, never a trained classifier target — positives are far too sparse (30,177 of 78.8M rows warehouse-wide) to trust a supervised label at this volume. I use it two ways: (a) to check whether pages that already show AI-referred sessions look different on the five features below, and (b) as the thing my ranked score is validated against, with **lift@K**, not AUC.

**Answer 5 — one thing I deliberately exclude.**
I exclude **any GA4 session total that is not already split out from AI-referred sessions** (i.e. an aggregate GA4 "all sessions" column, if the table ships one) from the feature side. `sessions_ai` sessions are a subset of a page's total GA4 sessions in the same measurement system — using a total-sessions column as a *feature* to rank pages by AI-referral opportunity would be near-circular (the label is partially baked into the input), even though it is not literally derived from the label the way `trend_pct` was in notebook 02. I would rather lose a plausibly useful signal than keep one that is adjacent to leakage.

**Full field classification for this contract:**

| Field | Bucket | Why |
|---|---|---|
| `report_date`, `client_hash_id`, `content_hash_id` | Context | Grain + join keys only; pseudonyms carry no signal on their own |
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` | Feature | Search Console measurements, independent of GA4/AI tracking, known day-by-day |
| `word_count`, `content_type` (`dim_content`) | Feature | Static content metadata, fixed at publish time, long before this month |
| `ga4_data_available` | Context (filter only) | Three-valued availability flag — used to filter, never fed to a model |
| `sessions_ai` | Label / proxy source | Rolled up into `has_ai_sessions_month`, the evidence variable — never a feature |
| any GA4 all-session total | Excluded | Overlaps the same measurement system as `sessions_ai` — see Answer 5 |


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3a. Three verification queries
1. **Grain** — does one row really mean report_date × client × content?
2. **Counts + date span** — how big is my slice, and does it match the `month=2026-03` window I claimed?
3. **Availability** — filtered with `IS TRUE` (per the data dictionary's warning that the availability flag is three-valued: `TRUE` / `FALSE` / `NULL`, so `= FALSE` or `NOT ...` silently mishandles the `NULL` rows), how many rows survive?

In [5]:
# Query 1 -- GRAIN: one row really is report_date x client x content (zero rows back == grain holds)
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_daily_month']}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print(f"duplicate-grain rows found: {len(grain_check)}  (0 means the stated grain holds)")
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate-grain rows found: 0  (0 means the stated grain holds)


,report_date,client_hash_id,content_hash_id,c


In [6]:
# Query 2 -- COUNTS + DATE SPAN of my slice
counts = con.sql(f"""
    SELECT
        COUNT(*)                          AS n_rows,
        MIN(report_date)                  AS min_date,
        MAX(report_date)                  AS max_date,
        COUNT(DISTINCT content_hash_id)   AS n_content_items,
        COUNT(DISTINCT client_hash_id)    AS n_clients
    FROM {TABLES['fact_daily_month']}
""").df()
counts


,n_rows,min_date,max_date,n_content_items,n_clients
0,9841378,2026-03-01,2026-03-31,331437,55


In [7]:
# Query 3 -- AVAILABILITY, filtered with IS TRUE (never "= FALSE" / "NOT ..." -- the flag is three-valued)
availability = con.sql(f"""
    SELECT
        COUNT(*) AS all_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)                         AS ga4_available_rows,
        ROUND(SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) * 100.0
              / COUNT(*), 2)                                                                 AS pct_ga4_available,
        SUM(CASE WHEN ga4_data_available IS TRUE AND sessions_ai > 0 THEN 1 ELSE 0 END)      AS rows_with_ai_sessions
    FROM {TABLES['fact_daily_month']}
""").df()
availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,all_rows,ga4_available_rows,pct_ga4_available,rows_with_ai_sessions
0,9841378,413966.0,4.21,5534.0


### 3b. Five features, max — built from the same `month=2026-03` slice

I aggregate the daily grain up to one row per content item for the month, then join static metadata from `dim_content`. `has_ai_sessions_month` rides along in the same view **only** as the evidence variable for the trap below — it is never one of the five features.

| # | Feature | Available when? |
|---|---|---|
| 1 | `total_gsc_impressions_month` | Knowable because it is summed from Search Console impressions, recorded daily throughout the month by a measurement system entirely separate from GA4/AI-referral tracking |
| 2 | `avg_gsc_position_month` | Same Search Console system — a page's average ranking position is known independent of whether anyone ever clicked through from an AI tool |
| 3 | `days_with_impressions_month` | A count of days GSC recorded any impressions — again a pure Search Console measurement, no dependency on AI-referral outcomes |
| 4 | `word_count` | A static content attribute set when the article was written, fixed long before this month's traffic happened |
| 5 | `content_type` | A structural category assigned at publish time — does not change with a given month's performance |


In [8]:
# Build the month-level content feature frame (daily grain -> one row per content item)
con.sql(f"""
    CREATE OR REPLACE TEMP VIEW content_month AS
    SELECT
        f.content_hash_id,
        ANY_VALUE(f.client_hash_id)                                                  AS client_hash_id,
        SUM(f.gsc_impressions)                                                       AS total_gsc_impressions_month,
        AVG(CASE WHEN f.gsc_impressions > 0 THEN f.gsc_avg_position END)             AS avg_gsc_position_month,
        COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0 THEN f.report_date END)       AS days_with_impressions_month,
        -- label-derived column, kept ONLY for the deliberate leak experiment in 3c, never a feature:
        SUM(CASE WHEN f.ga4_data_available IS TRUE THEN f.sessions_ai ELSE 0 END)    AS sessions_ai_month,
        MAX(CASE WHEN f.ga4_data_available IS TRUE AND f.sessions_ai > 0
                 THEN 1 ELSE 0 END)                                                  AS has_ai_sessions_month
    FROM {TABLES['fact_daily_month']} f
    GROUP BY 1
""")

features_df = con.sql(f"""
    SELECT
        cm.content_hash_id,
        cm.total_gsc_impressions_month,
        cm.avg_gsc_position_month,
        cm.days_with_impressions_month,
        dc.word_count,
        dc.content_type,
        cm.sessions_ai_month,       -- kept aside for 3c only
        cm.has_ai_sessions_month    -- evidence variable, not a feature
    FROM content_month cm
    JOIN {TABLES['dim_content']} dc USING (content_hash_id)
    -- minimum-volume filter so the ranking isn't dominated by near-zero-traffic noise:
    WHERE cm.total_gsc_impressions_month >= 100
""").df()

print(f"{len(features_df):,} content items in the demand-worthy month=2026-03 slice")
print(f"base rate of has_ai_sessions_month in this slice: {features_df['has_ai_sessions_month'].mean():.2%}")
features_df.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

101,441 content items in the demand-worthy month=2026-03 slice
base rate of has_ai_sessions_month in this slice: 3.16%


,content_hash_id,total_gsc_impressions_month,avg_gsc_position_month,days_with_impressions_month,word_count,content_type,sessions_ai_month,has_ai_sessions_month
0,content_cec711b02f3bbde6,602.0,4.428747,29,2455,keyword article,0.0,0
1,content_275b6f7f733016d4,810.0,4.866123,29,3653,keyword article,0.0,0
2,content_755d951187fcd70a,1858.0,1.854929,30,3305,keyword article,0.0,0
3,content_92c381fbd361212e,536.0,4.442543,29,3248,keyword article,0.0,0
4,content_97188a7032a705cf,496.0,4.018509,29,3309,keyword article,0.0,0


### 3c. The trap — add one label-derived column on purpose

I build an honest quick-score from the five features above, check its lift@K against `has_ai_sessions_month`, then deliberately fold in `sessions_ai_month` — a column summed straight from the label source — and watch the score jump toward perfect. Then I delete it and keep only the honest number, exactly the leakage lesson from notebook 02, performed here on real warehouse data.


In [9]:
import numpy as np

df = features_df.copy()
df["word_count"] = df["word_count"].fillna(0)

def rank01(s):
    """percentile rank, higher = more opportunity-like"""
    return s.rank(pct=True)

base_rate = df["has_ai_sessions_month"].mean()
K = max(50, int(0.05 * len(df)))  # top 5%, floor of 50

# --- HONEST quick score: only the five features from 3b, position inverted (lower number = better rank) ---
df["quick_score_honest"] = (
    rank01(df["total_gsc_impressions_month"])
    + rank01(-df["avg_gsc_position_month"].fillna(df["avg_gsc_position_month"].max()))
    + rank01(df["days_with_impressions_month"])
    + rank01(df["word_count"])
) / 4

top_k_honest = df.sort_values("quick_score_honest", ascending=False).head(K)
lift_honest = top_k_honest["has_ai_sessions_month"].mean() / base_rate

print(f"base rate of has_ai_sessions_month:      {base_rate:.2%}")
print(f"HONEST quick-score lift@{K}:              {lift_honest:.2f}x")

# --- THE TRAP: fold in sessions_ai_month, a column summed straight from the label source ---
df["quick_score_leaky"] = df["quick_score_honest"] + rank01(df["sessions_ai_month"])
top_k_leaky = df.sort_values("quick_score_leaky", ascending=False).head(K)
lift_leaky = top_k_leaky["has_ai_sessions_month"].mean() / base_rate

print(f"LEAKY quick-score lift@{K}:               {lift_leaky:.2f}x   <- jumps toward perfect, means nothing")
print("Why: sorting by a column summed from sessions_ai just re-sorts by (a rounding of) the label itself.")

# --- delete the leaky column and the leaky score, keep only the honest number ---
df = df.drop(columns=["sessions_ai_month", "quick_score_leaky"])
print(f"\nLeaky column removed. Honest number I keep: lift@{K} = {lift_honest:.2f}x")


base rate of has_ai_sessions_month:      3.16%
HONEST quick-score lift@5072:              1.20x
LEAKY quick-score lift@5072:               19.64x   <- jumps toward perfect, means nothing
Why: sorting by a column summed from sessions_ai just re-sorts by (a rounding of) the label itself.

Leaky column removed. Honest number I keep: lift@5072 = 1.20x


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** `has_ai_sessions_month` is a **same-month evidence variable**, not a forecast. This contract never claims to predict *future* AI-referral sessions — it only ranks existing demand-worthy pages by how closely they resemble pages that already show AI-referred sessions this month. Two things follow from that, both worth saying out loud:

- **Unbalanced panel.** `month=2026-03` sits inside a panel where per-client history depth differs wildly (`dim_clients.gsc_data_start` / `ga4_data_start`). A content item showing `has_ai_sessions_month = 0` may reflect thin or absent GA4 tracking for that client rather than a genuine absence of AI-referral opportunity — I filter on `ga4_data_available IS TRUE` above precisely to reduce this confound, but it does not remove it entirely.
- **Sparse positives, small counts.** AI-session rows are thin warehouse-wide (30,177 of 78.8M daily rows). Within one month, for one lane, the count of `has_ai_sessions_month = 1` content items will be small — the lift@K numbers above are directional evidence, not a statistically robust estimate, and should not be over-read as precise.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.